
# Clase 6 — La forma de los datos manda: texto, secuencias y embeddings

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Para que una máquina lea, cada palabra
> necesita un lugar en un mapa, y ese lugar se lo gana por la compañía que
> frecuenta.

---

**Cuando termines este notebook vas a poder:**

1. Convertir un corpus de texto en números: tokenizar, armar el vocabulario,
   contar coocurrencias.
2. Entrenar tus propios *embeddings* sobre tu corpus y buscar los vecinos de
   una palabra.
3. Medir el parecido entre dos documentos, y saber por qué el primer número
   que te va a dar está mal.
4. Usar un embedding como **instrumento de medición** de cómo se habla en un
   corpus — que es la parte que le sirve a tu investigación.

**Lo que hace falta traer:** las clases 1 a 4. Nada más. El *training loop* de
la clase 4 aparece de nuevo, sin un solo cambio.

**El corpus.** 35 discursos presidenciales de apertura de sesiones ordinarias
del Congreso argentino, 1984-2019. Son **reales** y son documentos públicos.
Están en `datos/muestras/discursos-apertura/corpus-mini-gpt.txt` y el README de
`datos/` cuenta sus trampas: los agujeros del corpus completo son los golpes de
Estado, y 2005 y 2006 son el mismo discurso.

**Cómo se usa esto.** Corré las celdas en orden (`Shift + Enter`). Donde diga
`# TODO` te toca a vos. Corre entero en **menos de un minuto** en CPU y **sin
internet**.


## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el
repo; en tu máquina no hace casi nada. Corrémela y seguí de largo.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys

REPO = "https://github.com/johannperez/dlss"
RAMA = "claude/dl-course-social-scientists-279dd2"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                        REPO, CARPETA], check=True)
    raiz = buscar_raiz(CARPETA)

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab: revisá que la celda haya podido clonar (necesita internet).\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`."
    )

# Dependencias: en Colab casi todas vienen puestas. Instalamos sólo lo que falte.
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],
                       check=True)

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")


### Imports y semilla

Fijamos la semilla en 1983 **siempre**. No hace al modelo mejor: hace a tu
resultado reproducible, que es condición para poder afirmar algo.

In [ ]:

import re
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt

from ssdl import datos, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso
utils.resumen_del_entorno()


---

## 1. El corpus, antes de tocarlo

Regla de la casa: **primero mirás el dato, después lo procesás.** Un corpus
que no leíste con los ojos es un corpus que te va a mentir.

In [ ]:

# El corpus completo, como un solo string. Si no está descargado, `ssdl` cae
# solo a la muestra que viene versionada en el repo: nunca falla por falta de red.
corpus = datos.corpus_mini_gpt()

# El archivo separa los discursos con una línea "### 1984 — Raul Alfonsin".
# La partimos para poder trabajar discurso por discurso más adelante.
partes = re.split(r"^###\s+(\d{4})\s+—\s+(.+)$", corpus, flags=re.M)
por_anio = {int(partes[i]): partes[i + 2] for i in range(1, len(partes), 3)}
anios = sorted(por_anio)

print(f"{len(corpus):,} caracteres · {len(por_anio)} discursos "
      f"({anios[0]}-{anios[-1]})")
print()
print("Un pedacito del discurso de 1984:")
print(por_anio[1984][240:560].strip())
print()
print("Y uno de 2019:")
print(por_anio[2019][330:650].strip())


> **Apostá antes de seguir.** ¿Cuántas *palabras distintas* creés que hay en
> estos 35 discursos? Escribí un número en un papel. Lo vamos a chequear en dos
> celdas.

---

## 2. Tokenizar: partir el texto en pedacitos

*Tokenizar* suena técnico y es "partir en pedacitos". La decisión importante no
es cómo se llama: es **qué cuenta como un pedazo**. Acá lo hacemos con una
expresión regular de una línea, a la vista, para que se entienda que es una
decisión nuestra y no una ley de la naturaleza.

Fijate lo que esta línea decide por vos:

- pasa todo a minúsculas → `Estado` y `estado` se vuelven la misma palabra;
- se queda sólo con letras → los números y `COVID-19` desaparecen;
- corta por espacios → `no fue aprobado` son tres fichas, y la negación queda
  como una ficha suelta que después pesa lo mismo que cualquier otra.

Ninguna de esas tres cosas es obviamente correcta. Son tres decisiones de
medición, como el libro de códigos de un análisis de contenido.

In [ ]:

def partir(texto):
    """Parte un texto en palabras: minúsculas, sólo letras (con tildes y ñ)."""
    return re.findall(r"[a-záéíóúñü]+", texto.lower())


fichas = partir(corpus)

print("Los primeros 30 pedacitos del discurso de 1984:")
print(partir(por_anio[1984])[:30])
print()
print(f"{len(fichas):,} fichas en total")
print(f"{len(set(fichas)):,} palabras distintas")


### El vocabulario, y una ley que nadie escribió

Contar es lo primero. `Counter` te da la tabla de frecuencias en una línea.

In [ ]:

frecuencias = Counter(fichas)

print("Las 15 palabras más frecuentes:")
for palabra, veces in frecuencias.most_common(15):
    print(f"  {palabra:12s} {veces:6,}")

print()
print("Y las 15 más frecuentes que NO son palabras de función:")
FUNCION = set("de la que y el en a los las del se por un una con no para es su "
              "al lo como más pero sus le ya o este sí porque esta entre cuando "
              "muy sin sobre también me hasta hay donde nos todos todo ha han he "
              "fue ser son era les ni".split())
mostradas = 0
for palabra, veces in frecuencias.most_common(400):
    if palabra in FUNCION or len(palabra) <= 3:
        continue
    print(f"  {palabra:14s} {veces:6,}")
    mostradas += 1
    if mostradas == 15:
        break

print()
print("Esa lista FUNCION la escribimos nosotros, a mano, doce líneas más arriba.")
print("Cambiarla cambia esta tabla. Como todo léxico: es el instrumento de")
print("medición, y no es neutral. (Clase 1, con datos nuevos.)")

In [ ]:

# La ley de Zipf: si ordenás las palabras de más a menos frecuente y graficás
# frecuencia contra puesto en escala log-log, sale una recta. Pasa en todos los
# idiomas y en todos los corpus, y nadie lo diseñó así.
cuentas = np.array(sorted(frecuencias.values(), reverse=True))
puestos = np.arange(1, len(cuentas) + 1)

fig, ax = g.figura(7.5, 4.6,
                   titulo="La ley de Zipf en los discursos de apertura",
                   xlabel="puesto en el ranking de frecuencia",
                   ylabel="veces que aparece")
ax.loglog(puestos, cuentas, color=g.DATOS, lw=1.8, label="lo observado")
ax.loglog(puestos, cuentas[0] / puestos, color=g.VERDAD, ls="--", lw=1.4,
          label="una recta perfecta (1 / puesto)")
ax.legend()
plt.show()

print(f"La palabra más frecuente aparece {cuentas[0]:,} veces.")
print(f"La palabra número 1000 aparece {cuentas[999]:,} veces.")


> **Por qué esto importa y no es un adorno.** La recta de Zipf dice que un
> puñado de palabras se lleva casi todo el texto y que **la mayoría del
> vocabulario aparece poquísimas veces**. Eso tiene una consecuencia práctica
> brutal: de la mayor parte de las palabras de tu corpus vas a tener muy poca
> evidencia, y cualquier cosa que midas sobre ellas va a ser ruido.

### `TODO 1` — ¿cuántas palabras aparecen una sola vez?

Se llaman *hapax legomena* ("dicho una sola vez"). Antes de calcularlo,
**adiviná el porcentaje**.

*Pista: `frecuencias` es un diccionario `palabra -> cuántas veces`. Contá
cuántos valores valen exactamente 1.*

In [ ]:

# TODO 1: contá cuántas palabras del vocabulario aparecen UNA SOLA VEZ.
#
# Reemplazá el None por tu cuenta. Una línea alcanza.
# Pista: sum(1 for veces in frecuencias.values() if ...)

hapax = None          # <- tu respuesta acá

# ---------------------------------------------------------------------------
if hapax is None:
    print("Todavía no lo resolviste: reemplazá el None de arriba por tu cuenta.")
else:
    esperado = sum(1 for v in frecuencias.values() if v == 1)
    if hapax != esperado:
        print(f"Te dio {hapax:,} y la cuenta da {esperado:,}.")
        print("Fijate que lo que hay que contar son PALABRAS DISTINTAS cuya")
        print("frecuencia vale exactamente 1, no la cantidad total de fichas.")
    else:
        print(f"✓ {hapax:,} palabras aparecen una sola vez.")
        print(f"  Son el {100 * hapax / len(frecuencias):.1f} % del vocabulario.")
        print()
        print("  Casi la mitad del vocabulario aparece UNA vez. De esas palabras")
        print("  no se puede aprender nada: no hay con qué. Por eso todo lo que")
        print("  viene después trabaja con las 2.000 más frecuentes y tira el resto.")
        print("  Tirar el resto no es gratis: mirá qué se pierde.")
        print("  Diez ejemplos de palabras que aparecen una sola vez:")
        raras = [p for p, v in frecuencias.items() if v == 1][:10]
        print("   ", ", ".join(raras))


---

## 3. La bolsa de palabras, y exactamente dónde se rompe

*Bag-of-words* (bolsa de palabras) representa un documento por **cuántas veces
aparece cada palabra**, sin orden. Es la base de los diccionarios, del análisis
de contenido cuantitativo y de LDA. Funcionó durante treinta años y sigue
funcionando: si tu pregunta es *de qué se habla*, contar palabras anda bárbaro.

El problema aparece con otra pregunta.

> **Las dos frases del arranque de la clase** eran *«la comisión rechazó al
> ministro»* y *«el ministro rechazó a la comisión»*. Con este tokenizador esas
> dos bolsas **no** salen exactamente iguales: cambian los artículos (`al` de un
> lado, `a` + `el` del otro). Ninguna palabra con contenido se mueve, y cualquier
> pipeline que saque las palabras vacías las deja idénticas. Acá usamos un par
> donde no hace falta ningún asterisco: dos sustantivos masculinos, y las dos
> bolsas salen iguales tal cual, sin tocar nada.

In [ ]:

frase_a = "el gobierno criticó al congreso"
frase_b = "el congreso criticó al gobierno"

bolsa_a = Counter(partir(frase_a))
bolsa_b = Counter(partir(frase_b))

print("frase A:", dict(bolsa_a))
print("frase B:", dict(bolsa_b))

assert bolsa_a == bolsa_b, "deberían ser idénticas"
print()
print("Las dos tablas de conteos son IDÉNTICAS. No parecidas: idénticas.")
print("Ningún método que represente un texto contando palabras puede")
print("distinguir estas dos frases. Y la diferencia entre las dos es")
print("quién criticó a quién, que para un análisis político es TODO.")


---

## 4. Coocurrencias: el mapa, a mano y sin ningún modelo

La **hipótesis distribucional** (Firth, 1957) dice: *"conocerás una palabra por
la compañía que mantiene"*. Es una tesis fuerte y discutible sobre el
significado — la discutimos en clase — y además es algo que se puede **contar**.

Contemos: para cada palabra, con qué otras aparece cerca (a menos de 5 palabras
de distancia). Eso es una matriz de coocurrencia. Sin modelos, sin entrenar
nada, sin redes: un doble `for` que ya está escrito en `ssdl.datos`.

In [ ]:

# ventana=5: "cerca" quiere decir a menos de 5 palabras a cada lado.
# minimo=8:  una palabra tiene que aparecer al menos 8 veces para entrar.
# top=2000:  nos quedamos con las 2.000 más frecuentes (acordate del hapax).
M, palabras = datos.matriz_de_coocurrencia(corpus, top=2000, ventana=5, minimo=8)
indice = {p: i for i, p in enumerate(palabras)}

print(f"Matriz de {M.shape[0]} x {M.shape[1]}. "
      f"M[i, j] = cuántas veces palabras[j] apareció cerca de palabras[i].")
print()


def acompanantes(palabra, k=8, sacar_funcion=True):
    """Las k palabras que más veces aparecen cerca de `palabra`."""
    fila = M[indice[palabra]]
    orden = np.argsort(-fila)
    salida = []
    for j in orden:
        if palabras[j] == palabra:
            continue
        if sacar_funcion and palabras[j] in FUNCION:
            continue
        salida.append((palabras[j], int(fila[j])))
        if len(salida) >= k:
            break
    return salida


for w in ["seguridad", "educación", "pobreza", "provincias", "mujeres"]:
    print(f"{w:12s} ->", ", ".join(f"{p} ({n})" for p, n in acompanantes(w, 6)))


Pará un segundo acá. **Esto ya es un resultado, y no hay ningún modelo
todavía.** Es un conteo. La hipótesis distribucional funcionando a ojo desnudo:
las palabras que van juntas cuentan una historia sobre cómo se habla de cada
tema en este corpus.

Los *embeddings* que vamos a entrenar en un rato **no son magia: son esto,
comprimido**. Toda la parte difícil ya la hiciste.

### `TODO 2` — tu palabra

In [ ]:

# TODO 2: elegí una palabra que te interese de tu área y mirá con quién anda.
#
# Ojo con dos cosas:
#   · las tildes cuentan: está "educación", no "educacion";
#   · tiene que estar entre las 2.000 más frecuentes.

mi_palabra = "justicia"        # <- cambiala por la tuya

if mi_palabra not in indice:
    print(f"«{mi_palabra}» no está en el vocabulario de {len(palabras)} palabras.")
    print("No es un error tuyo: quiere decir que aparece menos de 8 veces en")
    print("todo el corpus, o que la escribiste sin tilde. Probá con otra.")
    parecidas = [p for p in palabras if p.startswith(mi_palabra[:4])][:8]
    if parecidas:
        print("Palabras del corpus que empiezan parecido:", ", ".join(parecidas))
else:
    print(f"«{mi_palabra}» aparece {frecuencias[mi_palabra]:,} veces y anda con:")
    for p, n in acompanantes(mi_palabra, 10):
        print(f"   {p:16s} {n:4d} veces cerca")
    print()
    print("Escribí acá abajo, en una oración, qué te dice esa lista.")


**Tu interpretación:** _(escribí acá)_

---

## 5. De contar a un mapa: PPMI y SVD

La matriz de coocurrencia tiene dos problemas obvios:

1. **Es enorme**: 2.000 × 2.000. Una fila por palabra, con 2.000 números.
2. **Está dominada por lo frecuente**: `de` aparece cerca de todo, no porque
   tenga una relación especial con nada, sino porque `de` está en todos lados.

Los dos se arreglan con dos pasos, y ninguno es difícil:

- **PPMI** (*positive pointwise mutual information*) responde: ¿estas dos
  palabras aparecen juntas **más de lo que esperarías por puro azar**?
  Si `de` y `seguridad` aparecen juntas mucho pero las dos aparecen mucho con
  todo, el valor baja. Es la misma idea que un residuo de una tabla de
  contingencia: observado contra esperado.
- **SVD** se queda con las 100 direcciones que más variación explican. Es
  exactamente el gesto de un **análisis de correspondencias**: si ya hiciste
  ACM o ACP sobre una tabla de contingencia, ya hiciste esto.

Lo hacemos primero **a mano** sobre un pedacito, para que se vea que no hay
truco.

In [ ]:

# --- PPMI a mano, sobre las 800 palabras más frecuentes ---------------------
M_chica, pal_chicas = datos.matriz_de_coocurrencia(corpus, top=800, ventana=5,
                                                   minimo=8)
total = M_chica.sum()
p_fila = M_chica.sum(axis=1, keepdims=True) / total     # cuán frecuente es i
p_col = M_chica.sum(axis=0, keepdims=True) / total      # cuán frecuente es j

with np.errstate(divide="ignore", invalid="ignore"):
    # log( observado / esperado-si-fueran-independientes )
    pmi = np.log((M_chica / total) / (p_fila * p_col))

PPMI = np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=0.0)
PPMI = np.maximum(PPMI, 0.0)        # la P de PPMI: los negativos se van a cero

print(f"PPMI: {PPMI.shape}, valores de {PPMI.min():.2f} a {PPMI.max():.2f}")
print()
idx_c = {p: i for i, p in enumerate(pal_chicas)}
print("Antes (conteo crudo), quién acompaña a «seguridad»:")
fila = M_chica[idx_c["seguridad"]]
print("  ", [pal_chicas[j] for j in np.argsort(-fila)[:6]])
print("Después de PPMI (descontando lo que se explica por pura frecuencia):")
fila = PPMI[idx_c["seguridad"]]
print("  ", [pal_chicas[j] for j in np.argsort(-fila)[:6]])


Ahí está el trabajo que hace PPMI: antes de descontar, las acompañantes de
`seguridad` son `de`, `la`, `y`; después, palabras que dicen algo.

---

## 6. Los mismos vectores, con el loop de la clase 4

Ahora la parte linda. **No vamos a aprender nada nuevo sobre entrenamiento.**
Vamos a usar el mismo ciclo de cinco estaciones de la clase 4, sin cambiarle una
línea. Lo único que cambia es *qué dibuja el modelo*: antes dibujaba una curva
sobre unos puntos; hoy dibuja **un mapa de palabras**.

La idea: buscar una matriz `E` de 800 × 30 (un vector de 30 números por palabra)
y otra `C` igual, tales que `E @ C.T` se parezca lo más posible a la matriz
PPMI. O sea: *comprimir 800 números por palabra en 30, perdiendo lo menos
posible*. Las perillas son los 48.000 números de `E` y `C`, y las mueve el
descenso por gradiente.

In [ ]:

objetivo = torch.tensor(PPMI, dtype=torch.float32)
n = objetivo.shape[0]

gen = torch.Generator().manual_seed(1983)
E = (torch.randn(n, 30, generator=gen) * 0.1).requires_grad_(True)   # las palabras
C = (torch.randn(n, 30, generator=gen) * 0.1).requires_grad_(True)   # los contextos

opt = torch.optim.Adam([E, C], lr=0.02)

historial = []
for epoca in range(500):
    prediccion = E @ C.T                                  # 1. PREDECIR
    perdida = ((prediccion - objetivo) ** 2).mean()       # 2. COMPARAR
    opt.zero_grad()                                       # 3. BORRAR
    perdida.backward()                                    # 4. ANOTAR
    opt.step()                                            # 5. MOVER
    historial.append(perdida.item())

print(f"loss: {historial[0]:.4f}  ->  {historial[-1]:.4f}")
print(f"{utils.formatear_cantidad(E.numel() + C.numel())} perillas.")

fig, ax = g.figura(6.5, 3.4, titulo="Las mismas cinco líneas de la clase 4",
                   xlabel="época", ylabel="loss")
ax.plot(historial, color=g.MODELO, lw=2)
plt.show()


¿Y cómo sabemos que eso que salió sirve? Comparándolo con el atajo exacto.
El SVD hace la misma compresión **de una** y sin iterar. Si el descenso por
gradiente encontró algo parecido, las listas de vecinos deberían coincidir en
buena medida.

In [ ]:

def normalizar_filas(X):
    """Cada fila a norma 1, así el producto punto ES la similitud coseno."""
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)


V_descenso = normalizar_filas(E.detach().numpy())

U, S, Wt = np.linalg.svd(PPMI, full_matrices=False)   # el atajo exacto
V_svd = normalizar_filas(U[:, :30] * S[:30])


def vecinas(V, palabra, k=6):
    sim = V @ V[idx_c[palabra]]
    return [pal_chicas[j] for j in np.argsort(-sim)[1:k + 1]]


coincidencias = []
for w in ["seguridad", "educación", "trabajo", "inflación", "mujeres", "justicia"]:
    a, b = vecinas(V_descenso, w), vecinas(V_svd, w)
    coincidencias.append(len(set(a) & set(b)))
    print(f"{w}")
    print(f"   descenso por gradiente: {', '.join(a)}")
    print(f"   SVD (el atajo exacto) : {', '.join(b)}")
    print(f"   -> coinciden {coincidencias[-1]} de 6")
    print()

print(f"Promedio: {np.mean(coincidencias):.1f} de 6 vecinas en común.")
print("Dos caminos distintos, casi el mismo mapa. El descenso no es magia:")
print("es una manera de llegar a algo que también se puede calcular de una.")


---

## 7. Los embeddings del corpus entero, y los vecinos

Ya lo hiciste a mano. Ahora usamos la versión de `ssdl`, que hace exactamente
los tres pasos de recién (coocurrencia → PPMI → SVD) sobre las 2.000 palabras y
con 100 dimensiones. Son **unos cuatro segundos** en CPU (lo medimos: entre 4 y 5) y no baja nada de
internet.

> **Esta es la misma configuración que usa la visualización
> `viz/espacio-de-embeddings.html`.** Los vecinos que ves acá son exactamente
> los que ves en la pantalla del aula.

In [ ]:

with utils.medir_tiempo("entrenar los embeddings del corpus entero"):
    V, palabras = datos.embeddings_ppmi_svd(corpus, dim=100, top=2000,
                                            ventana=5, minimo=8)

indice = {p: i for i, p in enumerate(palabras)}
print(f"{V.shape[0]} palabras, {V.shape[1]} números cada una.")
print(f"Norma de un vector cualquiera: {np.linalg.norm(V[0]):.3f} "
      f"(están normalizados: el producto punto ya es la similitud coseno)")
print()

for w in ["pobreza", "seguridad", "docentes", "provincias", "mujeres"]:
    vs = datos.vecinos(w, V, palabras, k=6)
    print(f"{w:12s} ->", ", ".join(f"{p} ({s:.2f})" for p, s in vs))

print()
print("Mirá los vecinos de «seguridad»: se cuelan «de», «la», «través».")
print("No es un bug: PPMI descuenta la frecuencia pero no la elimina, y esas")
print("palabras están cerca de todo. En la práctica se filtran a mano — o sea,")
print("otra decisión tuya que hay que declarar en el paper.")


### `TODO 3` — los vecinos de tus palabras

Buscá los vecinos de **tres** palabras que te interesen. La consigna real no es
correr la celda: es **anotar cuál te sorprendió y por qué**. Esa sorpresa es un
dato sobre el corpus, y a veces es el principio de un paper.

*Pista: probá con palabras que tengan carga política, y probá también con una
palabra que creas neutra. Las neutras suelen dar la sorpresa más grande.*

In [ ]:

# TODO 3: cambiá esta lista por tres palabras tuyas.
mis_palabras = ["derechos", "crecimiento", "familia"]

for w in mis_palabras:
    if w not in indice:
        print(f"«{w}» no está en el vocabulario (aparece menos de 8 veces,")
        print("  o le falta una tilde). Probá con otra.")
        continue
    vs = datos.vecinos(w, V, palabras, k=10)
    print(f"\n{w}  (aparece {frecuencias[w]:,} veces)")
    for p, s in vs:
        print(f"   {p:18s} {s:.3f}")


**Cuál te sorprendió y por qué:** _(escribí acá)_

### El mapa, dibujado

Los vectores tienen 100 dimensiones y nosotros vemos dos. Así que proyectamos
con PCA y miramos la sombra.

**Decilo en voz alta cada vez que muestres uno de estos gráficos:** esto es una
sombra. Dos palabras pueden verse pegadas en el dibujo y estar lejísimos en las
100 dimensiones reales. El pie del gráfico dice qué porcentaje de la variación
estás viendo, y suele ser deprimente.

In [ ]:

# Elegimos 120 palabras con contenido (sin palabras de función), de las más
# frecuentes. OJO: esa lista FUNCION la escribimos nosotros más arriba, a mano.
# No es neutral, y cambiarla cambia el dibujo. Como todo léxico.
seleccion = [i for i, w in enumerate(palabras[:600])
             if w not in FUNCION and len(w) > 4][:120]

# y nos aseguramos de que las que queremos resaltar estén en el dibujo
a_resaltar = ["seguridad", "educación", "trabajo", "inversión"]
for w in a_resaltar:
    if indice[w] not in seleccion:
        seleccion.append(indice[w])

ax = g.proyeccion_de_embeddings(
    V[seleccion], [palabras[i] for i in seleccion],
    resaltar=a_resaltar,
    titulo="El mapa donde viven las palabras (una sombra de 100 dimensiones)")
plt.show()


---

## 8. Medir el parecido entre dos documentos

Esto es lo que más se usa en investigación: tenés dos textos y querés un número
que diga cuánto se parecen. La receta ingenua es promediar los vectores de sus
palabras y calcular el coseno.

**Hacela y mirá lo que pasa.** El resultado es una lección.

In [ ]:

def vector_de_documento(texto):
    """El vector de un documento = el promedio de los vectores de sus palabras."""
    fichas_doc = [f for f in partir(texto) if f in indice]
    v = V[[indice[f] for f in fichas_doc]].mean(axis=0)
    return v / np.linalg.norm(v)


Docs = np.array([vector_de_documento(por_anio[a]) for a in anios])
Sim = Docs @ Docs.T                      # la matriz de similitudes coseno

fuera_diagonal = Sim[~np.eye(len(Sim), dtype=bool)]
print(f"Similitud entre discursos distintos:")
print(f"   mínima {fuera_diagonal.min():.3f} · máxima {fuera_diagonal.max():.3f}"
      f" · promedio {fuera_diagonal.mean():.3f}")
print()
print("Todos los pares dan entre 0,97 y 1,00. Todo se parece a todo.")
print("Si reportaras 'la similitud entre 1984 y 2019 es 0,98' parecería")
print("altísima, y no significa nada: 0,98 es el piso, no el techo.")


**Por qué pasa.** Todos los discursos están hechos casi de las mismas palabras
frecuentes (`de`, `la`, `que`, `argentinos`, `país`). El promedio de cada
documento queda dominado por ese fondo común, y el fondo común es igual para
todos. Estás midiendo "los dos están escritos en castellano".

**El arreglo es de una línea y es el mismo de siempre en estadística: sacale la
media.** Restá el documento promedio y quedate con en qué se aparta cada uno.

In [ ]:

Centrados = Docs - Docs.mean(axis=0)                       # sacar el fondo común
Centrados = Centrados / np.linalg.norm(Centrados, axis=1, keepdims=True)
Sim_centrada = Centrados @ Centrados.T

fig, ejes = g.figura(12.5, 5, filas=1, columnas=2,
                     titulo="Parecido entre los 35 discursos de apertura")
for ax, Mx, t, lim in [(ejes[0], Sim, "promedio crudo: todo igual a todo", (0, 1)),
                       (ejes[1], Sim_centrada,
                        "restando el discurso promedio: aparece la estructura",
                        (-1, 1))]:
    im = ax.imshow(Mx, cmap="magma", vmin=lim[0], vmax=lim[1])
    ax.set_title(t, fontsize=11)
    ax.set_xticks(range(0, len(anios), 5))
    ax.set_xticklabels([anios[q] for q in range(0, len(anios), 5)], rotation=90)
    ax.set_yticks(range(0, len(anios), 5))
    ax.set_yticklabels([anios[q] for q in range(0, len(anios), 5)])
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

fuera = Sim_centrada.copy()
np.fill_diagonal(fuera, -9)
i, j = np.unravel_index(np.argmax(fuera), fuera.shape)
fuera2 = Sim_centrada + np.eye(len(anios)) * 9
k, l = np.unravel_index(np.argmin(fuera2), fuera2.shape)
print(f"Los dos discursos más parecidos:  {anios[i]} y {anios[j]}  "
      f"({Sim_centrada[i, j]:.3f})")
print(f"Los dos más distintos:            {anios[k]} y {anios[l]}  "
      f"({Sim_centrada[k, l]:.3f})")


### `TODO 4` — el del lunes

Mirá el panel de la derecha. Se ven **bloques**: períodos en los que los
discursos se parecen entre sí y se diferencian de los otros.

1. ¿Dónde están los cortes? Anotá los años.
2. ¿Coinciden con lo que sabés de la historia política argentina de ese
   período? ¿Con cambios de gobierno, con crisis, con otra cosa?
3. Los dos discursos más parecidos y los dos más distintos que imprimió la
   celda: ¿te sorprenden?

**Este es el momento del notebook donde tu conocimiento sustantivo vale más que
el código.** El modelo encontró bloques; qué significan esos bloques no lo puede
decir. Y ojo con el atajo fácil: que dos discursos se parezcan puede querer
decir que hablan de lo mismo, que los escribió el mismo equipo, o que en los dos
años pasaba lo mismo en el país. El número no distingue entre esas tres cosas.

*Pista: mirá la fila de 2001 y la de 2002 y compará con sus vecinas.*

In [ ]:

# TODO 4: usá esta celda para mirar de cerca los años que te llamaron la atención.
# Cambiá los dos años y corré.

anio_1, anio_2 = 2001, 2010

i1, i2 = anios.index(anio_1), anios.index(anio_2)
print(f"Parecido entre {anio_1} y {anio_2}: {Sim_centrada[i1, i2]:.3f}")
print()

# ¿Qué palabras tiene uno mucho más que el otro? (una comparación de conteos,
# la herramienta más vieja del análisis de contenido, y sigue sirviendo)
c1 = Counter(f for f in partir(por_anio[anio_1]) if f in indice)
c2 = Counter(f for f in partir(por_anio[anio_2]) if f in indice)
t1, t2 = sum(c1.values()), sum(c2.values())
difs = {w: c1[w] / t1 - c2[w] / t2 for w in set(c1) | set(c2)
        if (c1[w] + c2[w]) > 25 and w not in FUNCION}
orden = sorted(difs, key=difs.get)

print(f"Mucho más presentes en {anio_1}:")
print("  ", ", ".join(reversed(orden[-12:])))
print(f"Mucho más presentes en {anio_2}:")
print("  ", ", ".join(orden[:12]))


**Tu lectura de los bloques:** _(escribí acá)_

---

## 9. Sesgo, o instrumento de medición

Esta sección es la más importante del notebook y la que más vale para tu
investigación.

Un embedding aprendió **cómo se usa cada palabra en este corpus**. Todas las
regularidades de uso, no sólo las que nos gustan. Si en treinta y cinco años de
discursos ciertas palabras aparecieron sistemáticamente cerca de otras, eso
quedó grabado en la geometría.

La técnica: elegís dos palabras que definan una **dirección** (un eje) y
proyectás el vocabulario sobre ella. Eso es todo. Es la metodología de
Kozlowski, Taddy & Evans (2019) en la *American Sociological Review*, y de
Garg et al. (2018) en *PNAS*.

In [ ]:

def proyectar_sobre_eje(polo_positivo, polo_negativo, lista):
    """Ordena `lista` según de qué lado del eje cae cada palabra."""
    for w in (polo_positivo, polo_negativo):
        if w not in indice:
            raise KeyError(f"«{w}» no está en el vocabulario de {len(palabras)} "
                           f"palabras. Ojo con las tildes.")
    eje = V[indice[polo_positivo]] - V[indice[polo_negativo]]
    eje = eje / np.linalg.norm(eje)
    salida = [(w, float(V[indice[w]] @ eje)) for w in lista if w in indice]
    return sorted(salida, key=lambda t: -t[1])


conceptos = ["familia", "hogar", "hijos", "niños", "cuidado", "escuela",
             "salud", "casa", "trabajo", "salario", "empresa", "violencia",
             "seguridad", "poder", "fuerza", "derechos"]

ranking = proyectar_sobre_eje("mujeres", "hombres", conceptos)

faltan = [w for w in conceptos if w not in indice]
if faltan:
    print("Fuera del vocabulario (no entran al ranking):", ", ".join(faltan))
    print()

print("Eje  hombres  <---->  mujeres\n")
for w, v in ranking:
    barra = "█" * int(abs(v) * 60)
    lado = "mujeres" if v > 0 else "hombres"
    print(f"  {w:12s} {v:+.3f}  {barra:<20s} {lado if abs(v) > 0.05 else ''}")


> **Cómo NO leer esto.** El número no dice "esta palabra es femenina". Dice:
> *en este corpus, esta palabra aparece en contextos más parecidos a los de
> «mujeres» que a los de «hombres»*. Es una afirmación sobre 35 discursos
> presidenciales argentinos entre 1984 y 2019, y sobre nada más.
>
> Y ojo con el tamaño: los valores van de −0,12 a +0,32. Son chicos porque el
> eje es una dirección entre cien. **Lo que se interpreta es el orden, no la
> magnitud.**

### `TODO 5` — la pregunta de criterio

Este resultado, ¿es **un defecto del modelo** o **un dato sobre el corpus**?

Escribí tres oraciones respondiendo:

1. ¿De dónde viene ese orden?
2. Si usaras este embedding para decidir a quién contratar, ¿qué pasaría?
3. Si lo usaras para **estudiar** cómo cambió el discurso oficial sobre el
   cuidado entre 1984 y 2019, ¿qué pasaría?

*Pista: la respuesta cambia según para qué vayas a usar el modelo. No es una
propiedad del modelo, es una propiedad del uso.*

**Tu respuesta:** _(escribí acá)_

In [ ]:

# Probá otros ejes. Cualquier par de palabras del vocabulario define uno.
# Algunos que dan cosas interesantes en este corpus:
for polo_pos, polo_neg in [("estado", "mercado"), ("futuro", "pasado")]:
    print(f"\nEje  {polo_neg}  <---->  {polo_pos}")
    r = proyectar_sobre_eje(polo_pos, polo_neg, conceptos)
    print(f"   más cerca de «{polo_pos}»: ",
          ", ".join(f"{w} ({v:+.2f})" for w, v in r[:4]))
    print(f"   en el otro extremo:      ",
          ", ".join(f"{w} ({v:+.2f})" for w, v in reversed(r[-4:])))


---

## 10. La advertencia que te va a salvar de un revisor

Es la confusión número uno con embeddings, y aparece en papers publicados:

> **Cerca en el mapa NO quiere decir que signifiquen lo mismo.**
> Quiere decir que **se usan parecido**.

Y hay un montón de palabras que se usan parecidísimo y significan lo contrario.

In [ ]:
pares = [("importaciones", "exportaciones"), ("público", "privado"),
         ("aumento", "reducción"), ("gobierno", "oposición"),
         ("guerra", "paz"), ("mayoría", "minoría")]

print("Pares de palabras opuestas, y su similitud coseno:\n")
for a, b in pares:
    if a in indice and b in indice:
        s = float(V[indice[a]] @ V[indice[b]])
        print(f"  {a:16s} vs {b:16s}  {s:.3f}")
    else:
        falta = a if a not in indice else b
        print(f"  {a:16s} vs {b:16s}  (falta «{falta}» en el vocabulario)")

print()
print("Las cuatro primeras son altísimas: para el espacio, son casi la misma")
print("palabra. Y mirá las dos últimas, porque son la parte honesta:")
print()
print("· «guerra»/«paz» es EL ejemplo clásico de este argumento, y acá NO da alto.")
print("  En 35 discursos de apertura de la democracia argentina, «paz» aparece en")
print("  contextos de convivencia interna y «guerra» casi no aparece. El fenómeno")
print("  es real; que se vea depende del corpus. Si hubiéramos mostrado sólo los")
print("  cuatro pares que funcionan, te habríamos vendido algo.")
print("· «minoría» ni siquiera entró al vocabulario: aparece menos de 8 veces.")
print("  El corte de frecuencia también decide qué podés medir y qué no.")


### `TODO 6` — el ejercicio conceptual más importante del notebook

`importaciones` y `exportaciones` son operaciones opuestas y su similitud es
altísima. `público` y `privado` también.

1. ¿Por qué quedan cerca?
2. Si usaras la similitud coseno entre documentos como medida de **acuerdo
   político** entre dos discursos, ¿qué estarías midiendo en realidad?
3. Nombrá una pregunta de investigación de **tu** área en la que esta confusión
   te arruinaría el resultado.

*Pista para (1): pensá en qué oraciones aparecen las dos palabras de cada par.
Fijate que un antónimo es la palabra que MÁS fácil se cambia por la otra sin
que la oración deje de tener sentido.*

**Tu respuesta:** _(escribí acá)_

---

## 11. Qué te llevás

**Las tres cosas de hoy:**

1. **Para que una máquina lea, cada palabra necesita un lugar en el espacio, y
   ese lugar sale de con quién aparece.** Coocurrencia → PPMI → SVD. Lo hiciste
   a mano y después con el mismo loop de la clase 4: no había magia adentro.
2. **La geometría de ese mapa es un instrumento de medición.** Un eje definido
   por dos palabras ordena el vocabulario, y ese orden es un dato sobre cómo se
   habló en tu corpus. Eso es un paper de la *ASR*, no un truco.
3. **Cerca quiere decir "se usan parecido", no "significan lo mismo".** Los
   antónimos están cerquísima. Y el orden de las palabras se pierde entero:
   promediar vectores no distingue quién le hizo qué a quién.

**El vocabulario nuevo** (también en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *token* | un pedacito de texto: casi siempre una palabra, a veces un pedazo de palabra |
| *vocabulary* | la lista de todos los pedacitos que el modelo conoce |
| *OOV* (out of vocabulary) | una palabra que no está en esa lista, y que el modelo no ve |
| *one-hot* | un vector de ceros con un solo 1: un casillero por palabra |
| *embedding* | los pocos números que ubican a una palabra en un mapa |
| *cosine similarity* | qué tan parecidos apuntan dos vectores, de −1 a 1 |
| *bag-of-words* | representar un texto por sus conteos, tirando el orden |
| *distributional hypothesis* | "conocerás una palabra por la compañía que mantiene" |
| *RNN* | red que lee en orden arrastrando un resumen de lo anterior |

**Para el lunes, en tu propia investigación.** Agarrá tu corpus (si tiene más de
unos 300.000 tokens, alcanza), corré las tres líneas de la sección 7, y mirá los
vecinos de las cinco palabras que definen tu objeto de estudio. Vas a tardar
diez minutos y vas a aprender algo sobre tu corpus que no sabías. Después, si
querés más, armá un eje con dos palabras y proyectá una lista de términos: eso
ya es una medición reportable, siempre que digas de qué corpus salió.

Si tu corpus es chico (200 documentos cortos), **no entrenes embeddings
propios**: te van a salir ruido. Ahí van modelos pre-entrenados, que es la
clase 9.

**Lo que queda prometido.** Promediar vectores pierde el orden, y una memoria
que se arrastra se olvida del principio. La clase que viene: en vez de recordar,
**mirar directamente**. Se llama *attention*.

---

### Para seguir

Las referencias están en `referencias.md`. Si vas a leer una sola cosa,
que sea Rodriguez & Spirling (2022): es la guía práctica escrita para
exactamente esta audiencia, y responde la pregunta de cuándo entrenar
embeddings propios y cuándo usar pre-entrenados.